# Block 03: OF Baseline, CRB, and Real Rank-1 Verification

**Goal**  
Use the real K-alpha dataset to anchor rank-1 equivalence, bridge diagnostics, and observed-vs-predicted amplitude spread.

**Checklist alignment**  
Implements the real-data side of E5 and the full real verification target E12.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
out = run_block03_real_rank1(cfg)
summary = pd.Series(out["summary"], name="value").to_frame()
display(summary)
display(out["bridge_df"])

In [ ]:
amp_df = out["amplitude_df"]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(amp_df["of_gls"], amp_df["empca_rank1"], s=10, alpha=0.5)
lo = min(amp_df["of_gls"].min(), amp_df["empca_rank1"].min())
hi = max(amp_df["of_gls"].max(), amp_df["empca_rank1"].max())
ax.plot([lo, hi], [lo, hi], color="black", lw=1.0, ls="--")
ax.set_xlabel("OF amplitude (GLS)")
ax.set_ylabel("EMPCA rank-1 amplitude")
ax.set_title("Real K-alpha: OF vs rank-1 EMPCA")
save_figure(fig, dirs["figures"] / "block_03_real_rank1_scatter.png")
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
vals = amp_df["of_gls"].to_numpy()
ax.hist(vals, bins=30, density=True, alpha=0.7, label="OF amplitudes")
mu = vals.mean()
sigma = vals.std(ddof=1)
xs = np.linspace(vals.min(), vals.max(), 300)
ax.plot(xs, stats.norm.pdf(xs, loc=mu, scale=sigma), color="black", lw=1.5, label="Gaussian fit")
ax.set_xlabel("amplitude")
ax.set_ylabel("density")
ax.set_title("Real K-alpha amplitude histogram")
ax.legend()
save_figure(fig, dirs["figures"] / "block_03_real_amplitude_histogram.png")
plt.close(fig)

In [ ]:
summary_path = dirs["tables"] / "block_03_e12_real_summary.csv"
amp_path = dirs["tables"] / "block_03_real_rank1_amplitudes.csv"
bridge_path = dirs["tables"] / "block_03_real_bridge.csv"
manifest_path = dirs["manifests"] / "block_03_real_summary.json"
save_dataframe(pd.DataFrame([out["summary"]]), summary_path)
save_dataframe(out["amplitude_df"], amp_path)
save_dataframe(out["bridge_df"], bridge_path)
save_json({"summary": out["summary"], "of_stats": out["of_stats"]}, manifest_path)
pd.DataFrame(
    [
        manifest_row("block_03", "real-support", str(summary_path.relative_to(REPO_ROOT)), cfg, out["bundle"]),
        manifest_row("block_03", "real-support", str(bridge_path.relative_to(REPO_ROOT)), cfg, out["bundle"]),
    ]
)